# Launch and set up AMD MI100 server - with python-chi

In [1]:
from chi import server, context, lease
import os, time
import chi

context.version = "1.0" 
context.choose_project()
context.choose_site(default="CHI@TACC")

In [2]:
l = lease.get_lease(f"node1_project35") 
l.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>node1_project3…

Lease Details:
Name: node1_project35
ID: b535112f-2859-44c7-945d-1272ced09cec
Status: ACTIVE
Start Date: 2025-05-11 18:35:00
End Date: 2025-05-11 19:55:00
User ID: b47b677115e1dbbb5a28b1e0aba16c88c2a1b3f108f89e257536d6d2c5a56379
Project ID: d3c6e101843a4ba79e665ebf59b521a2

Node Reservations:
ID: e8d9d1ca-e430-4598-889b-85a5a4c4280e, Status: active, Min: 1, Max: 1

Floating IP Reservations:

Network Reservations:

Events:


In [3]:
username = "group35"
s = server.Server(
    f"node-mltrain-{username}", 
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-hwe"
)
s.submit(idempotent=True)

Waiting for server node-mltrain-group35's status to become ACTIVE. This typically takes 10 minutes, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-mltrain-group35
Id,fbf78cdf-ee64-438e-be86-ee7adf32fd83
Status,ACTIVE
Image Name,CC-Ubuntu24.04-hwe
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.2.101 (v4) Type: fixed MAC: 34:80:0d:de:52:98
Network Name,sharednet1
Created At,2025-05-11T18:41:27Z
Keypair,ym2985_nyu_edu-jupyter
Reservation Id,e8d9d1ca-e430-4598-889b-85a5a4c4280e
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


In [4]:
s.associate_floating_ip()

In [5]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.108.211 port 22.


Connection successful


In [6]:
s.refresh()
s.show(type="widget")

Attribute,node-mltrain-group35
Id,fbf78cdf-ee64-438e-be86-ee7adf32fd83
Status,ACTIVE
Image Name,CC-Ubuntu24.04-hwe
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.2.101 (v4) Type: fixed MAC: 34:80:0d:de:52:98 IP: 129.114.108.211 (v4) Type: floating MAC: 34:80:0d:de:52:98
Network Name,sharednet1
Created At,2025-05-11T18:41:27Z
Keypair,ym2985_nyu_edu-jupyter
Reservation Id,e8d9d1ca-e430-4598-889b-85a5a4c4280e
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


In [7]:
# security_groups = [
#   {'name': "allow-ssh", 'port': 22, 'description': "Enable SSH traffic on TCP port 22"},
#   {'name': "allow-8888", 'port': 8888, 'description': "Enable TCP port 8888 (used by Jupyter)"},
#   {'name': "allow-8000", 'port': 8000, 'description': "Enable TCP port 8000 (used by FastAPI)"},
#   {'name': "allow-9000", 'port': 9000, 'description': "Enable TCP port 9000 (used by MinIO API)"},
#   {'name': "allow-9001", 'port': 9001, 'description': "Enable TCP port 9001 (used by MinIO Web UI)"},
#   {'name': "allow-9090", 'port': 9090, 'description': "Enable TCP port 9090 (used by Prometheus)"},
#   {'name': "allow-5000", 'port': 5000, 'description': "Enable TCP port 9001 (used by MLFlow)"},
#   {'name': "allow-3000", 'port': 3000, 'description': "Enable TCP port 9001 (used by Grafana)"},
# ]

# os_conn = chi.clients.connection()
# nova_server = chi.nova().servers.get(s.id)

# for sg in security_groups:
#   nova_server.add_security_group(sg['name'])

# print(f"updated security groups: {[group.name for group in nova_server.list_security_group()]}")

## Retrieve code and notebooks on the instance

In [8]:
s.execute("git clone --recurse-submodules https://github.com/M0n4GPT/vision-to-vintage")

/opt/conda/lib/python3.10/site-packages/paramiko/client.py:889: UserWarning: Unknown ssh-ed25519 host key for 129.114.108.211: b'3b9a0def12d928a59dc30be41d3e0a13'
  warnings.warn(
Cloning into 'vision-to-vintage'...


<Result cmd='git clone --recurse-submodules https://github.com/M0n4GPT/vision-to-vintage' exited=0>

In [9]:
s.execute("mv vision-to-vintage/style_transfer ./style_transfer")
s.execute("rm -rf vision-to-vintage")

<Result cmd='rm -rf vision-to-vintage' exited=0>

## Set up Docker
To use common deep learning frameworks like Tensorflow or PyTorch, and ML training platforms like MLFlow and Ray, we can run containers that have all the prerequisite libraries necessary for these frameworks. Here, we will set up the container framework.

In [10]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

# Executing docker install script, commit: 53a22f61c0628e58e1d6680b49e82993d304b449


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin >/dev/null

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c doc

Client: Docker Engine - Community
 Version:           28.1.1
 API version:       1.49
 Go version:        go1.23.8
 Git commit:        4eba377
 Built:             Fri Apr 18 09:52:14 2025
 OS/Arch:           linux/amd64
 Context:           default

Server: Docker Engine - Community
 Engine:
  Version:          28.1.1
  API version:      1.49 (minimum version 1.24)
  Go version:       go1.23.8
  Git commit:       01f442b
  Built:            Fri Apr 18 09:52:14 2025
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          1.7.27
  GitCommit:        05044ec0a9a75232cad458027ca83437aae3f4da
 runc:
  Version:          1.2.5
  GitCommit:        v1.2.5-0-g59923ef
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0


To run Docker as a non-privileged user, consider setting up the
Docker daemon in rootless mode for your user:

    dockerd-rootless-setuptool.sh install

Visit https://docs.docker.com/go/rootless/ to learn about rootless mode.


T

<Result cmd='sudo groupadd -f docker; sudo usermod -aG docker $USER' exited=0>

## Set up the AMD GPU
Before we can use the AMD GPUs, we need to set up the driver using the amdgpu-install utility.

In [11]:
s.execute("sudo apt update; wget https://repo.radeon.com/amdgpu-install/6.3.3/ubuntu/noble/amdgpu-install_6.3.60303-1_all.deb")
s.execute("sudo apt -y install ./amdgpu-install_6.3.60303-1_all.deb; sudo apt update")

Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:4 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1067 kB]
Get:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/main amd64 Components [161 kB]
Get:8 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [1063 kB]
Get:9 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/universe amd64 Components [376 kB]
Get:10 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Components [212 B]
Get:11 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Components [940 B]
Get:12 http://nova.clouds.archiv

--2025-05-11 18:54:48--  https://repo.radeon.com/amdgpu-install/6.3.3/ubuntu/noble/amdgpu-install_6.3.60303-1_all.deb
Resolving repo.radeon.com (repo.radeon.com)... 23.221.22.215, 23.221.22.214, 2600:1404:6400:25::17de:f154, ...
Connecting to repo.radeon.com (repo.radeon.com)|23.221.22.215|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16984 (17K) [application/octet-stream]
Saving to: ‘amdgpu-install_6.3.60303-1_all.deb’

     0K .......... ......                                     100% 37.4M=0s

2025-05-11 18:54:48 (37.4 MB/s) - ‘amdgpu-install_6.3.60303-1_all.deb’ saved [16984/16984]





Reading package lists...
Building dependency tree...
Reading state information...
Recommended packages:
  dialog
The following NEW packages will be installed:
  amdgpu-install
0 upgraded, 1 newly installed, 0 to remove and 145 not upgraded.
Need to get 0 B/17.0 kB of archives.
After this operation, 74.8 kB of additional disk space will be used.
Get:1 /home/cc/amdgpu-install_6.3.60303-1_all.deb amdgpu-install all 6.3.60303-2119913.24.04 [17.0 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Selecting previously unselected package amdgpu-install.
(Reading database ... 93276 files and directories currently installed.)
Preparing to unpack .../amdgpu-install_6.3.60303-1_all.deb ...
Unpacking amdgpu-install (6.3.60303-2119913.24.04) ...
Setting up amdgpu-install (6.3.60303-2119913.24.04) ...


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.




Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Get:2 https://repo.radeon.com/amdgpu/6.3.3/ubuntu noble InRelease [5435 B]
Get:3 https://repo.radeon.com/rocm/apt/6.3.3 noble InRelease [2605 B]
Get:4 https://repo.radeon.com/amdgpu/6.3.3/ubuntu noble/main amd64 Packages [14.1 kB]
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:6 https://repo.radeon.com/amdgpu/6.3.3/ubuntu noble/main i386 Packages [12.2 kB]
Get:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:8 https://repo.radeon.com/rocm/apt/6.3.3 noble/main amd64 Packages [60.0 kB]
Get:9 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Fetched 603 kB in 1s (533 kB/s)
Reading package lists...
Building dependency tree...
Reading state information...
145 packages can be upgraded. Run 'apt list --upgradable' to see them.


<Result cmd='sudo apt -y install ./amdgpu-install_6.3.60303-1_all.deb; sudo apt update' exited=0>

In [12]:
s.execute("amdgpu-install -y --usecase=dkms")
s.execute("sudo apt -y install rocm-smi")
s.execute("sudo usermod -aG video,render $USER")

Hit:1 https://repo.radeon.com/amdgpu/6.3.3/ubuntu noble InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease
Hit:3 https://repo.radeon.com/rocm/apt/6.3.3 noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:5 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Fetched 508 kB in 1s (457 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
linux-headers-6.11.0-17-generic is already the newest version (6.11.0-17.17~24.04.2).
linux-headers-6.11.0-17-generic set to manually installed.
The following additional packages will be installed:
  amdgpu-dkms-firmware autoconf automake autotools-dev m4
Suggested packages:
  autoconf-archive gnu-standards autoconf-doc libtool gettext m4-doc
The following N

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 27.9 MB in 1s (23.5 MB/s)
Selecting previously unselected package m4.
(Reading database ... 93294 files and directories currently installed.)
Preparing to unpack .../0-m4_1.4.19-4build1_amd64.deb ...
Unpacking m4 (1.4.19-4build1) ...
Selecting previously unselected package autoconf.
Preparing to unpack .../1-autoconf_2.71-3_all.deb ...
Unpacking autoconf (2.71-3) ...
Selecting previously unselected package autotools-dev.
Preparing to unpack .../2-autotools-dev_20220109.1_all.deb ...
Unpacking autotools-dev (20220109.1) ...
Selecting previously unselected package automake.
Preparing to unpack .../3-automake_1%3a1.16.5-1.3ubuntu1_all.deb ...
Unpacking automake (1:1.16.5-1.3ubuntu1) ...
Selecting previously unselected package amdgpu-dkms-firmware.
Preparing to unpack .../4-amdgpu-dkms-firmware_1%3a6.10.5.60303-2119913.24.04_all.deb ...
Unpacking amdgpu-dkms-firmware (1:6.10.5.60303-2119913.24.04) ...
Selecting previously unselected package amdgpu-dkms.
Preparing to unpack .../5-am

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.




Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  librocm-smi64-1
The following NEW packages will be installed:
  librocm-smi64-1 rocm-smi
0 upgraded, 2 newly installed, 0 to remove and 145 not upgraded.
Need to get 362 kB of archives.
After this operation, 1744 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 librocm-smi64-1 amd64 5.7.0-1 [309 kB]
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 rocm-smi amd64 5.7.0-1 [52.9 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 362 kB in 1s (367 kB/s)
Selecting previously unselected package librocm-smi64-1.
(Reading database ... 97702 files and directories currently installed.)
Preparing to unpack .../librocm-smi64-1_5.7.0-1_amd64.deb ...
Unpacking librocm-smi64-1 (5.7.0-1) ...
Selecting previously unselected package rocm-smi.
Preparing to unpack .../rocm-smi_5.7.0-1_amd64.deb ...
Unpacking rocm-smi (5.7.0-1) ...
Setting up librocm-smi64-1 (5.7.0-1) ...
Setting up rocm-smi (5.7.0-1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.4) ...


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.


<Result cmd='sudo usermod -aG video,render $USER' exited=0>

In [13]:
s.execute("sudo reboot")
time.sleep(30)

In [14]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.108.211 port 22.


Connection successful


In [15]:
s.execute("rocm-smi")



========================= ROCm System Management Interface =========================
=================================== Concise Info ===================================
GPU  Temp (DieEdge)  AvgPwr  SCLK    MCLK     Fan  Perf  PwrCap  VRAM%  GPU%  
0    29.0c           34.0W   300Mhz  1200Mhz  0%   auto  290.0W    0%   0%    
1    27.0c           35.0W   300Mhz  1200Mhz  0%   auto  290.0W    0%   0%    
=============================== End of ROCm SMI Log ================================


<Result cmd='rocm-smi' exited=0>

and verify that you can see the GPU(s).

also install nvtop

In [16]:
s.execute("sudo apt -y install cmake libncurses-dev libsystemd-dev libudev-dev libdrm-dev libgtest-dev")
s.execute("git clone https://github.com/Syllo/nvtop")
s.execute("mkdir -p nvtop/build && cd nvtop/build && cmake .. -DAMDGPU_SUPPORT=ON && sudo make install")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  cmake-data googletest libdrm-amdgpu1 libdrm-intel1 libdrm-nouveau2
  libdrm-radeon1 libjsoncpp25 libnss-systemd libpam-systemd libpciaccess-dev
  libpciaccess0 librhash0 libsystemd-shared libsystemd0 libudev1 systemd
  systemd-dev systemd-resolved systemd-sysv udev
Suggested packages:
  cmake-doc cmake-format elpa-cmake-mode ninja-build ncurses-doc
  systemd-container systemd-homed systemd-userdbd systemd-boot libqrencode4
  libtss2-rc0
The following NEW packages will be installed:
  cmake cmake-data googletest libdrm-amdgpu1 libdrm-dev libdrm-intel1
  libdrm-nouveau2 libdrm-radeon1 libgtest-dev libjsoncpp25 libncurses-dev
  libpciaccess-dev libpciaccess0 librhash0 libsystemd-dev libudev-dev
The following packages will be upgraded:
  libnss-systemd libpam-systemd libsystemd-shared libsystemd0 libudev1 systemd
  systemd-dev systemd-resolved systemd-sys

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 25.3 MB in 2s (11.2 MB/s)
(Reading database ... 97717 files and directories currently installed.)
Preparing to unpack .../systemd-dev_255.4-1ubuntu8.6_all.deb ...
Unpacking systemd-dev (255.4-1ubuntu8.6) over (255.4-1ubuntu8.5) ...
Preparing to unpack .../systemd-resolved_255.4-1ubuntu8.6_amd64.deb ...
Unpacking systemd-resolved (255.4-1ubuntu8.6) over (255.4-1ubuntu8.5) ...
Preparing to unpack .../libsystemd-shared_255.4-1ubuntu8.6_amd64.deb ...
Unpacking libsystemd-shared:amd64 (255.4-1ubuntu8.6) over (255.4-1ubuntu8.5) ...
Preparing to unpack .../libsystemd0_255.4-1ubuntu8.6_amd64.deb ...
Unpacking libsystemd0:amd64 (255.4-1ubuntu8.6) over (255.4-1ubuntu8.5) ...
Setting up libsystemd0:amd64 (255.4-1ubuntu8.6) ...
(Reading database ... 97717 files and directories currently installed.)
Preparing to unpack .../0-systemd-sysv_255.4-1ubuntu8.6_amd64.deb ...
Unpacking systemd-sysv (255.4-1ubuntu8.6) over (255.4-1ubuntu8.5) ...
Preparing to unpack .../1-libnss-systemd_255.4-1ubuntu

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

Restarting services...
 systemctl restart firewalld.service multipathd.service polkit.service rpcbind.service rsyslog.service ssh.service systemd-hostnamed.service systemd-timedated.service udisks2.service

Service restarts being deferred:
 systemctl restart ModemManager.service
 /etc/needrestart/restart.d/dbus.service
 systemctl restart docker.service
 systemctl restart networkd-dispatcher.service
 systemctl restart systemd-logind.service
 systemctl restart unattended-upgrades.service

No containers need to be restarted.

User sessions running out

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Setting build type to 'Release' as none was specified.
-- Looking for cbreak in /usr/lib/x86_64-linux-gnu/libncursesw.so
-- Looking for cbreak in /usr/lib/x86_64-linux-gnu/libncursesw.so - found
-- Found Curses: /usr/lib/x86_64-linux-gnu/libncursesw.so  
-- Performing Test HAS_REALLOCARRAY
-- Performing Test HAS_REALLOCARRAY - Success
-- Found UDev: /usr/lib/x86_64-linux-gnu/libudev.so (found version "") 
-- Libudev stable: FALSE
-- Found Systemd: /usr/lib/x86_64-linux-gnu/libsystemd.so 

<Result cmd='mkdir -p nvtop/build && cd nvtop/build && cmake .. -DAMDGPU_SUPPORT=ON && sudo make install' exited=0>

### Build a container image - for MLFlow section

Finally, we will build a container image in which to work in the MLFlow section, that has:

-   a Jupyter notebook server
-   Pytorch and Pytorch Lightning
-   ROCm, which allows deep learning frameworks like Pytorch to use the AMD GPU accelerator
-   and MLFlow

You can see our Dockerfile for this image at: [Dockerfile.jupyter-torch-mlflow-rocm](https://github.com/teaching-on-testbeds/mltrain-chi/tree/main/docker/Dockerfile.jupyter-torch-mlflow-rocm)

Building this container will take a **very long** time (ROCm is huge). But that’s OK: we can get it started and then continue to the next section while it builds in the background, since we don’t need this container immediately. We just need it to finish by the “Start a Jupyter server” subsection of the “Start the tracking server” section.

In [17]:
s.execute("docker build -t jupyter-mlflow -f style_transfer/docker/Dockerfile.jupyter-torch-mlflow-rocm .")

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile.jupyter-torch-mlflow-rocm
#1 transferring dockerfile: 1.16kB done
#1 DONE 0.0s

#2 [internal] load metadata for quay.io/jupyter/scipy-notebook:latest
#2 DONE 0.9s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/5] FROM quay.io/jupyter/scipy-notebook:latest@sha256:38b74c0b58d1e004bb979f5a221f5730578f9aca7f6878c1689f4a193c4793cc
#4 resolve quay.io/jupyter/scipy-notebook:latest@sha256:38b74c0b58d1e004bb979f5a221f5730578f9aca7f6878c1689f4a193c4793cc done
#4 sha256:0cec0952d04b35f63c37822bc4a9d3a83ae200dd5331aba8ef73c7e266ae1559 6.99kB / 6.99kB done
#4 sha256:96c932f29ab238a89357a1ed3185a558d6195bed23a42e3f7c8eec419dfec130 0B / 11.43MB 0.1s
#4 sha256:33fe43ff4ffc27673f783d85a90aa649ec18870589bbce3ded03d5ca65e62351 18.52kB / 18.52kB done
#4 sha256:ac0c285abb482df6684de5a61b4577fc5cc5fafe8cd1280ebf52d8909d121599 0B / 30.59MB 0.1s
#4 sha256:ac0c285

<Result cmd='docker build -t jupyter-mlflow -f style_transfer/docker/Dockerfile.jupyter-torch-mlflow-rocm .' exited=0>

Leave that cell running, and in the meantime，On the compute instance, install rclone:

Open an SSH sesson on your server. From your local terminal, run

```bash
ssh -i ~/.ssh/id_rsa_chameleon_35 cc@A.B.C.D
```

## Mount the object store to compute instance

Now that our data is safely inside the object store, we can use it anywhere - on a VM, on a bare metal site, on multiple compute instances at once, even outside of Chameleon - to train or evaluate a model. We would not have to repeat the ETL pipeline each time we want to use the data.

If we were working on a brand-new compute instance, we would need to download `rclone` and create the rclone configuration file at `~/.config/rclone.conf`



```bash
# run on node
curl https://rclone.org/install.sh | sudo bash
```
We also need to modify the configuration file for FUSE (Filesystem in USErspace: the interface that allows user space applications to mount virtual filesystems), so that object store containers mounted by our user will be availabe to others, including Docker containers:

```bash
# run on node
# this line makes sure user_allow_other is un-commented in /etc/fuse.conf
sudo sed -i '/^#user_allow_other/s/^#//' /etc/fuse.conf
```

Next, create a configuration file for rclone with the ID and secret from the application credential you just generated:

```bash
# run on node
mkdir -p ~/.config/rclone
nano  ~/.config/rclone/rclone.conf
```

Paste the following into the config file, but substitute your own application credential ID and secret.

You will also need to substitute your own user ID. You can find it using “Identity” > “Users” in the Horizon GUI; it is an alphanumeric string (not the human-readable user name).
```
[chi_tacc]
type = swift
user_id = b47b677115e1dbbb5a28b1e0aba16c88c2a1b3f108f89e257536d6d2c5a56379
application_credential_id = 27829a1ac66f487aaf82cd92bfa047cf
application_credential_secret = 2d2PCXeS6d4cXZDSc-yAsskYH26Rcx8CCv4yEhucMAFafL3_ljj8njNn3V-OSSauEIugdH8XjUMyEE0iTXWISQ
auth = https://chi.tacc.chameleoncloud.org:5000/v3
region = CHI@TACC
```
Use Ctrl+O and Enter to save the file, and Ctrl+X to exit `nano`.

To test it, run

```bash
# run on node27829a1ac66f487aaf82cd92bfa047cf
rclone lsd chi_tacc:
```

and verify that you see your container listed. This confirms that rclone can authenticate to the object store.


mkdir -p ~/project35The next step is to create a mount point for the data in the local filesystem:

```bash
# run on node
mkdir -p ~/project35
```

Now finally, we can use rclone mount to mount the object store at the mount point (substituting your own netID in the command below).

```bash
# run on node
rclone mount chi_tacc:object-persist-project35 ~/project35 \--read-only --allow-other --daemon
```
Here,

`chi_tacc` tells rclone which section of its configuration file to use for authentication information
`object-persist-project35` tells it what object store container to mount
`~/project35` says where to mount it
Since we only intend to read the data, we can mount it in read-only mode and it will be slightly faster; and we are also protected from accidental writes. We also specified `--allow-other` so that we can use the mount from Docker, and `--daemon` means the rclone process will be started in the background.

(If need to umount,just do `umount ~/project35`)

Run
```bash
# run on node
ls project35/
```

and confirm that we can now see the img data directories.
our data directories would be:
```
project35/train
project35/val
project35/test
project35/random_inputs/random_train
project35/random_inputs/random_val
project35/random_inputs/random_test
```


## Start the tracking server

The YAML configuration is at: docker/docker-compose-mlflow.yaml

### Start MLFlow tracking server system
Now we are ready to get it started! Bring up our MLFlow system with:

``` bash
# run on node-mltrain
docker compose -f style_transfer/docker/docker-compose-mlflow.yaml up -d
```

which will pull each container image, then start them.

When it is finished, the output of

``` bash
# run on node-mltrain
docker ps
```

should show that the `minio`, `postgres`, and `mlflow` containers are running.

### Access dashboards for the MLFlow tracking server system

Both MLFlow and MinIO include a browser-based dashboard. Let’s open these to make sure that we can find our way around them.

The MinIO dashboard runs on port 9001. In a browser, open

    http://A.B.C.D:9001

Log in with the credentials we specified in the Docker Compose YAML:

-   Username: `your-access-key`
-   Password: `your-secret-key`

Then,

-   Click on the “Buckets” section and note the `mlflow-artifacts` storage bucket that we created as part of the Docker Compose.
-   Click on “Monitoring \> Metrics” and note the dashboard that shows the storage system health. MinIO works as a distributed object store with many advanced capabilities, although we are not using them; this dashboard lets operators keep an eye on system status.
-   Click on “Object Browser”. In this section, you can look at the files that have been uploaded to the object store - but, we haven’t used MLFlow yet, so for now there is nothing interesting here. However, as you start to log artifacts to the MLFlow server, you will see them appear here.

Next, let’s look at the MLFlow UI. This runs on port 8000. In a browser, open

    http://A.B.C.D:8000

The UI shows a list of tracked “experiments”, and experiment “runs”. (A “run” corresponds to one instance of training a model; an “experiment” groups together related runs.) Since we have not yet used MLFlow, for now we will only see a “Default” experiment and no runs. But, that will change very soon!

### Start a Jupyter server

Finally, we’ll start the Jupyter server container, inside which we will run experiments that are tracked in MLFlow. Make sure your container image build, from the previous section, is now finished - you should see a “jupyter-mlflow” image in the output of:

``` bash
# run on node-mltrain
docker image list
```

The command to run will depend on what type of GPU node you are using -

If you are using an AMD GPU (node type `gpu_mi100`), run

modify the MLFLOW_TRACKING_URI if needed!!!

``` bash
# run on node-mltrain IF it is a gpu_mi100
HOST_IP=$(curl --silent http://169.254.169.254/latest/meta-data/public-ipv4 )
docker run  -d --rm  -p 8888:8888 \
    --device=/dev/kfd --device=/dev/dri \
    --group-add video --group-add $(getent group | grep render | cut -d':' -f 3) \
    --shm-size 16G \
    -v /home/cc/style_transfer:/home/jovyan/work \
    -e MLFLOW_TRACKING_URI=http://${HOST_IP}:8000/ \
    -e IMG_DATA_DIR=/project35 \
    --mount type=bind,source=/home/cc/project35,target=/project35,readonly \
    --name jupyter \
    jupyter-mlflow
```

Note that we intially get `HOST_IP`, the floating IP assigned to your instance, as a variable; then we use it to specify the `MLFLOW_TRACKING_URI` inside the container. Training jobs inside the container will access the MLFlow tracking server using its public IP address.

Then, run

    docker logs jupyter

and look for a line like

    http://127.0.0.1:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Paste this into a browser tab, but in place of `127.0.0.1`, substitute the floating IP assigned to your instance, to open the Jupyter notebook interface.

In the file browser on the left side, open the `work` directory.

Open a terminal (“File \> New \> Terminal”) inside the Jupyter server environment, and in this terminal, run

``` bash
# runs on jupyter container inside node-mltrain
env
```

to see environment variables. Confirm that the `MLFLOW_TRACKING_URI` is set, with the correct floating IP address.

train_style_transfer_50

### Run a training job with mlflow and access the data from object storage

Open a terminal inside this environment (“File \> New \> New Terminal”) and `cd` to the `work` directory.
``` bash
# run in a terminal inside jupyter container
cd ~/work
ls
```
In the `style_transfer` directory, open `train_style_transfer_50.py`, and view it directly there.

Then, run `train_style_transfer_50.py`:

``` bash
# run in a terminal inside jupyter container
python -m torch.distributed.launch \
  --nproc_per_node=2 \
  train_style_transfer_50.py \
  --data_root "$IMG_DATA_DIR" \
  --epochs 5 \
  --precision amp \
  --strategy ddp \
  --export_path ./stylizer.pt
```


or if fine tune:

``` bash
# run in a terminal inside jupyter container
python -m torch.distributed.launch \                                                                                                               --nproc_per_node=2 \
  train_style_transfer_50.py \
      --data_root "$IMG_DATA_DIR" \
      --epochs 3 \
      --precision amp \
      --strategy ddp \
      --fine_tune \
      --pretrained_model ./stylizer.pt \
      --export_path ./stylizer.pt
```

## Ray train 

please move to the ray_train.ipynb then

### Run a non-MLFlow training job

Open a terminal inside this environment (“File \> New \> New Terminal”) and `cd` to the `work` directory. Then, clone the [gourmetgram-train](https://github.com/teaching-on-testbeds/gourmetgram-train/) repository:

``` bash
# run in a terminal inside jupyter container
cd ~/work
ls
```

In the `style_transfer` directory, open `train_style_transfer.py`, and view it directly there.

Then, run `train_style_transfer.py`:

``` bash
# run in a terminal inside jupyter container
cd ~/work
python3 train_style_transfer.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 32 \
    --micro_batch_size 8 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt
```

(note that the location of the Food-11 dataset has been specified in an environment variable passed to the container.)

Don’t let it finish (it would take a long time) - this is just to see how it works, and make sure it doesn’t crash. Use Ctrl+C to stop it running after a few minutes.

### Add MLFlow logging to Pytorch code
### modify:diff!!!!

After working on this model for a while, though, you realize that you are not being very effective because it’s difficult to track, compare, version, and reproduce all of the experiments that you run with small changes. To address this, at the organization level, the ML Platform team at GourmetGram has set up a tracking server that all GourmetGram ML teams can use to track their experiments. Moving forward, your training scripts should log all the relevant details of each training run to MLFlow.

Switch to the `mlflow` branch of the `gourmetgram-train` repository:

``` bash
# run in a terminal inside jupyter container, from the "work/gourmetgram-train" directory
git fetch -a
git switch mlflow
```

The `train.py` script in this branch has already been augmented with MLFlow tracking code. Run the following to see a comparison betweeen the original and the modified training script.

``` bash
# run in a terminal inside jupyter container, from the "work/gourmetgram-train" directory
git diff main..mlflow
```

(press `q` after you have finished reviewing this diff.)

The changes include:

**Add imports for MLFlow**:

``` python
import mlflow
import mlflow.pytorch
```

MLFlow includes framework-specific modules for many machine learning frameworks, including [Pytorch](https://mlflow.org/docs/latest/python_api/mlflow.pytorch.html), [scikit-learn](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html), [Tensorflow](https://mlflow.org/docs/latest/python_api/mlflow.tensorflow.html), [HuggingFace/transformers](https://mlflow.org/docs/latest/python_api/mlflow.transformers.html), and many more. In this example, most of the functions we will use come from base `mlflow`, but we will use an `mlflow.pytorch`-specific function to save the Pytorch model.

**Configure MLFlow**:

The main configuration that is required for MLFlow tracking is to tell the MLFlow client where to send everything we are logging! By default, MLFlow assumes that you want to log to a local directory named `mlruns`. Since we want to log to a remote tracking server, you’ll have to override this default.

One way to specify the location of the tracking server would be with a call to `set_tracking_uri`, e.g.

``` python
mlflow.set_tracking_uri("http://A.B.C.D:8000/") 
```

where `A.B.C.D` is the IP address of your tracking server. However, we may prefer not to hard-code the address of the tracking server in our code (for example, because we may occasionally want the same code to log to different tracking servers).

In these experiments, we will instead specify the location of the tracking server with the `MLFLOW_TRACKING_URI` environment variable, which we have already passed to the container.

(A list of other environment variables that MLFLow uses is available in [its documentation](https://mlflow.org/docs/latest/python_api/mlflow.environment_variables.html). )

We also set the “experiment”. In MLFlow, an “experiment” is a group of related “runs”, e.g. different attempts to train the same type of model. If we don’t specify any experiment, then MLFlow logs to a “default” experiment; but we will specify that runs of this code should be organized inside the “food11-classifier” experiment.

``` python
mlflow.set_experiment("food11-classifier")
```

**Start a run**:

In MLFlow, each time we train a model, we start a new run. Before we start training, we call

``` python
mlflow.start_run()
```

or, we can put all the training inside a

``` python
with mlflow.start_run():
    # ... do stuff
```

block. In this example, we actually start a run inside a

``` python
try: 
    mlflow.end_run() # end pre-existing run, if there was one
except:
    pass
finally:
    mlflow.start_run()
```

block, since we are going to interrupt training runs with Ctrl+C, and without “gracefully” ending the run, we may not be able to start a new run.

**Track system metrics**:

Also, when we called `start_run`, we passed a `log_system_metrics=True` argument. This directs MLFlow to automatically start tracking and logging details of the host on which the experiment is running: CPU utilization and memory, GPU utilization and memory, etc.

Note that to automatically log GPU metrics, we must have installed `pyrsmi` (for AMD GPUs) or `pynvml` (for NVIDIA GPUs) - we installed these libraries inside the container image already. (But if we would build a new container image, we’d want to remember that.)

Besides for the details that are tracked automatically, we also decided to get the output of `rocm-smi` (for AMD GPUs) or `nvidia-smi` (for NVIDIA GPUs), and save the output as a text file in the tracking server. This type of logged item is called an artifact - unlike some of the other data that we track, which is more structured, an artifact can be any kind of file.

We used

``` python
mlflow.log_text(gpu_info, "gpu-info.txt")
```

to save the contents of the `gpu_info` variable as a text file artifact named `gpu-info.txt`.

**Log hyperparameters**:

Of course, we will want to save all of the hyperparameters associated with our training run, so that we can go back later and identify optimal values. Since we have already saved all of our hyperparameters as a dictionary at the beginning, we can just call

``` python
mlflow.log_params(config)
```

passing that entire dictionary. This practice of defining hyperparameters in one place (a dictionary, an external configuration file) rather than hard-coding them throughout the code, is less error-prone but also easier for tracking.

**Log metrics during training**:

Finally, the thing we most want to track: the metrics of our model during training! We use `mlflow.log_metrics` inside each training run:

``` python
mlflow.log_metrics(
{"epoch_time": epoch_time,
    "train_loss": train_loss,
    "train_accuracy": train_acc,
    "val_loss": val_loss,
    "val_accuracy": val_acc,
    "trainable_params": trainable_params,
    }, step=epoch)
```

to log the training and validation metrics per epoch. We also track the time per epoch (because we may want to compare runs on different hardware or different distributed training strategies) and the number of trainable parameters (so that we can sanity-check our fine tuning strategy).

**Log model checkpoints**:

During the second part of our fine-tuning, when we un-freeze the backbone/base layer, we log the same metrics. In this training loop, though, we additionally log a model checkpoint at the end of each epoch if the validation loss has improved:

``` python
mlflow.pytorch.log_model(food11_model, "food11")
```

The model *and* many details about it will be saved as an artifact in MLFlow.

**Log test metrics**:

At the end of the training run, we also log the evaluation on the test set:

``` python
mlflow.log_metrics(
    {"test_loss": test_loss,
    "test_accuracy": test_acc
    })
```

and finally, we finish our run with

``` python
mlflow.end_run()
```

### 1 Run Pytorch code with MLFlow logging

To test this code, open `train_style_transfer_mlflow.py` , change the `### Configure MLFlow
mlflow.set_tracking_uri("http://129.114.C.D:8000/") ` into proper floating ip, then run

``` bash
# run in a terminal inside jupyter container, from the "work/gourmetgram-train" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 32 \
    --micro_batch_size 8 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt
```

(Note that we already passed the `MLFLOW_TRACKING_URI` and `FOOD11_DATA_DIR` to the container, so we do not need to specify this environment variable again when launching the training script.)

While this is running, in another tab in your browser, open the URL

    http://A.B.C.D:8000/

where in place of `A.B.C.D`, substitute the floating IP address assigned to *your* instance. You will see the MLFlow browser-based interface. Now, in the list of experiments on the left side, you should see the “food11-classifier” experiment. Click on it, and make sure you see your run listed. (It will be assigned a random name, since we did not specify the run name.)

Click on your run to see an overview. Note that in the “Details” field of the “Source” table, the exact Git commit hash of the code we are running is logged, so we know exactly what version of our training script generated this run.

As the training script runs, you will see a “Parameters” table and a “Metrics” table on this page, populated with values logged from the experiment.

-   Look at the “Parameters” table, and note that the hyperparameters in the `config` dictionary, which we logged with `log_params`, are all there.
-   Look at the “Metrics” section, and note that (at least) the most recent value of each of the system metrics appear there. Once an epoch has passed, model metrics will also appear there.

Click on the “System metrics” tab for a visual display of the system metrics over time. In particular, look at the time series chart for the `gpu_0_utilization_percentage` metric, which logs the utilization of the first GPU over time. Wait until a few minutes of system metrics data has been logged. (You can use the “Refresh” button in the top right to update the display.)


## 2 speed up strategy, squeeze out more speed-ups:

change the `DataLoader` as follows, bumping up workers (and adding a couple more useful flags) to keep your GPUs fed:

```diff

-   loader = DataLoader(
-       ds,
-       batch_size=args.micro_batch_size,
-       sampler=sampler,
-       shuffle=not distributed,
-       num_workers=4,
-       pin_memory=True
-   )

+   loader = DataLoader(
+       ds,
+       batch_size=args.micro_batch_size,
+       sampler=sampler,
+       shuffle=not distributed,
+       num_workers=16,                # ↑ more parallelism
+       pin_memory=True,               # keep this on
+       prefetch_factor=2,             # number of batches to prefetch per worker
+   )
```

Now, run the training script again with

``` bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 32 \
    --micro_batch_size 8 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt

```

In the MLFlow interface, find this new run, and open its overview. Note that the commit hash associated with this updated code is logged. You can also write a note to yourself, to remind yourself later what the objective behind this experiment was; click on the pencil icon next to “Description” and then put text in the input field, e.g.

Checking if increasing num_workers helps bring up GPU utilization. then, click “Save”. 

Note the difference between these training runs in:

-   the utilization of GPU 0 (logged as `gpu_0_utilization_percentage`, under system metrics)
-   and the time per epoch (logged as `epoch_time`, under model metrics)

we should see that your system metrics logging has allowed us to **substantially** speed up training by realizing that the GPU utilization aws low, and taking steps to address it. At the end of the training run, you will save these two plot panels for your reference.

#  Large-scale model training
## Training strategies for large models

### 3 Experiment: Baseline

As a baseline, let's try an epoch of training style transfer model, using full precision and a batch size of 128:


``` bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 128 \
    --micro_batch_size 128 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt

```

This is using about 15.30GB GPU memoryfor each epoch.

### 4 Experiment: Reduced batch size

But with a smaller batch size, it fits easily:
Make a note of the training time and memory, which is printed at the end of the training job.

``` bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 32 \
    --micro_batch_size 32 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt

```

### 5 Experiment: Gradient accumulation

By using gradient accumulation to "step" only after a few "micro batches", we can train with a larger effective "global" batch size, with minimal effect on the memory required:
``` bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 128 \
    --micro_batch_size 32 \
    --epochs 5 \
    --precision fp32 \
    --strategy none \
    --export_path ./stylizer.pt

```

Make a note of the training time and memory, which is printed at the end of the training job.

### 6 Experiment: Mixed precision

With mixed precision, we get back some of the lost precision in the results, at the cost of some additional memory and time:

``` bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 train_style_transfer_mlflow.py \
    --data_root "$IMG_DATA_DIR" \
    --global_batch_size 128 \
    --micro_batch_size 32 \
    --epochs 5 \
    --precision amp \
    --strategy none \
    --export_path ./stylizer.pt

```

Make a note of the training time and memory, which is printed at the end of the training job.



## Train a large model on multiple GPUs - 2x gpu_mi100

### Confirm ROCm-built PyTorch & RCCL support

```bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
python3 - << 'EOF'
import torch
print("Build backend:", torch.version.hip or torch.version.cuda)
print("NCCL/RCCL available:", torch.distributed.is_nccl_available())
print("CUDA/ROCm available:", torch.cuda.is_available(),
      "Device count:", torch.cuda.device_count())
EOF

```
should see the output:
```
Build backend: 6.3.42131-fa1d09cbd
NCCL/RCCL available: True
CUDA/ROCm available: True Device count: 2
```

### Verify ROCm tools
```bash
# run in a terminal inside the jupyter container, from inside the "work/" directory
which rocm-smi && rocm-smi --showid
```
You should see GPU info, proving ROCm user-space tools are installed. See the output:
```

============================ ROCm System Management Interface ============================
=========================================== ID ===========================================
GPU[0]          : Device Name:          Arcturus GL-XL [Instinct MI100]
GPU[0]          : Device ID:            0x738c
GPU[0]          : Device Rev:           0x01
GPU[0]          : Subsystem ID:         0x0c34
GPU[0]          : GUID:                 51219
GPU[1]          : Device Name:          Arcturus GL-XL [Instinct MI100]
GPU[1]          : Device ID:            0x738c
GPU[1]          : Device Rev:           0x01
GPU[1]          : Subsystem ID:         0x0c34
GPU[1]          : GUID:                 45163
==========================================================================================
================================== End of ROCm SMI Log ===================================
```


### 7 Experiment: train model on 2x gpu_mi100 with DDP

After all the check above, we are ready to run strategies for training a large model using distributed processes across multiple GPUs

```bash
# run in a terminal inside the jupyter container, from inside the "work/" directory

python -m torch.distributed.launch \                                                                                                                          --nproc_per_node=2 train_style_transfer_muti.py \
      --data_root "$IMG_DATA_DIR" \
      --global_batch_size 128 \
      --micro_batch_size 32 \
      --epochs 5 \
      --precision amp \
      --strategy ddp \
      --export_path ./stylizer_ddp.pt
```

### 8 Experiment: train model on 2x gpu_mi100 with FSDP


With DDP, we have a larger effective batch size (since 2 GPUs process a batch in parallel), but no memory savings. With FSDP, we can shard optimizer state, gradients, and parameters across GPUs, to also reduce the memory required.

```bash
# run in a terminal inside the jupyter container, from inside the "work/" directory

python -m torch.distributed.launch \
      --nproc_per_node=2 train_style_transfer_muti.py \
  --data_root "$IMG_DATA_DIR" \
  --global_batch_size 128 \
  --micro_batch_size 32 \
  --epochs 5 \
  --precision amp \
  --strategy fsdp \
  --export_path ./stylizer_fsdp.pt
```

note:

Why there's only a tiny GPU‐memory win when you switch to FSDP:

Activations matter more: During training, intermediate feature maps (activations) typically consume far more GPU memory than the model weights, and FSDP does not shard those by default.

Frozen encoder bug: Early on, I froze the VGG-19 encoder (set `requires_grad=False`) and then wrapped the entire model in FSDP with `use_orig_params=False`, causing a “uniform requires_grad” error. To unblock that, you black-listed the encoder so FSDP only sharded the CNN decoder.

Encoder dominates the parameters: VGG-19 accounts for ~143 M of ~145 M parameters, while the decoder has much small parameter size. By ignoring the encoder, you left ~572 MB of weights fully replicated on each GPU and only shard a samll part.


### 9 Experiment: train model on 2x gpu_mi100 with FSDP (modified)


Change your FSDP wrap to:

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import CPUOffload, ShardingStrategy

# … including the diff
(This one doesn't work!!)

if args.strategy == 'fsdp':
    model = FSDP(
        model,
        cpu_offload=CPUOffload(False),
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        flatten_parameters=False,   # DO NOT collapse into one flat buffer
        use_orig_params=True,       # keep each nn.Parameter intact
        # ignored_modules={enc},   # do not shard the frozen encoder
    )
With these two flags:
Each Conv2d.weight stays a 4-D tensor, so conv2d calls work correctly.
All weights (including your frozen encoder!) get individually sharded across GPUs, so you still get the parameter-memory benefit.

```bash
# run in a terminal inside the jupyter container, from inside the "work/" directory

python -m torch.distributed.launch \
      --nproc_per_node=2 train_style_transfer_fsdp.py \
  --data_root "$IMG_DATA_DIR" \
  --global_batch_size 128 \
  --micro_batch_size 32 \
  --epochs 5 \
  --precision amp \
  --strategy fsdp \
  --export_path ./stylizer_fsdp_modi.pt
```